# 23CSE301 Machine Learning – Capstone Project

**Team No:** 8

**Project Title:** Healthcare Cost Prediction and Patient Risk Intelligence Platform

**Dataset:** Medical Insurance Cost Prediction Dataset (`medical_insurance.csv`)


In [4]:
# Clustering Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN,
    SpectralClustering
)

from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style='whitegrid')

np.random.seed(42)

print("Clustering libraries imported successfully.")

Clustering libraries imported successfully.


In [5]:
df_raw = pd.read_csv('../data/medical_insurance.csv')

print("Dataset loaded successfully.")
print("Shape:", df_raw.shape)

Dataset loaded successfully.
Shape: (100000, 54)


In [6]:
# Create a copy of the raw dataset
df_cluster = df_raw.copy()

# Remove target and leakage-related columns
columns_to_remove = [
    'annual_medical_cost',
    'is_high_risk',
    'risk_score'
]

df_cluster = df_cluster.drop(
    columns=[col for col in columns_to_remove if col in df_cluster.columns]
)

print("Clustering dataset shape:", df_cluster.shape)

Clustering dataset shape: (100000, 51)


In [7]:
# Handle missing values

# Fill categorical columns with their mode
for col in df_cluster.select_dtypes(include='str').columns:
    df_cluster[col] = df_cluster[col].fillna(
        df_cluster[col].mode()[0]
    )

# Fill numerical columns with their median
for col in df_cluster.select_dtypes(include=np.number).columns:
    df_cluster[col] = df_cluster[col].fillna(
        df_cluster[col].median()
    )

print("Remaining missing values:",
      df_cluster.isnull().sum().sum())

Remaining missing values: 0


In [8]:
# Convert categorical variables into numerical variables
df_cluster_encoded = pd.get_dummies(
    df_cluster,
    drop_first=True
)

print("Encoded data shape:", df_cluster_encoded.shape)

Encoded data shape: (100000, 70)


In [9]:
# Scale clustering data
cluster_scaler = StandardScaler()

X_cluster_scaled = cluster_scaler.fit_transform(
    df_cluster_encoded
)

print("Clustering data scaled successfully.")
print("Scaled data shape:", X_cluster_scaled.shape)

Clustering data scaled successfully.
Scaled data shape: (100000, 70)


In [10]:
# Clustering Models

models = {
    'K-Means': KMeans(n_clusters=4,random_state=42,n_init=10),

    'Agglomerative': AgglomerativeClustering(
        n_clusters=4,
        linkage='ward'
    ),

    'DBSCAN': DBSCAN(
        eps=0.5,
        min_samples=5
    ),

    'Gaussian Mixture': GaussianMixture(
        n_components=4,
        random_state=42
    ),

    'Spectral Clustering': SpectralClustering(
        n_clusters=4,
        random_state=42,
        affinity='nearest_neighbors'
    )
}

In [ ]:
# Train models and calculate clustering metrics

results = []

for name, model in models.items():

    labels = model.fit_predict(X_cluster_scaled)

    # Metrics require at least 2 clusters
    if len(set(labels)) > 1:

        silhouette = silhouette_score(
            X_cluster_scaled,
            labels
        )

        davies_bouldin = davies_bouldin_score(
            X_cluster_scaled,
            labels
        )

        calinski_harabasz = calinski_harabasz_score(
            X_cluster_scaled,
            labels
        )

        results.append({
            'Algorithm': name,
            'Silhouette Score': silhouette,
            'Davies-Bouldin Index': davies_bouldin,
            'Calinski-Harabasz Index': calinski_harabasz
        })

    else:
        print(name, 'produced only one cluster.')

print("Clustering metrics calculated successfully.")

In [ ]:
# Clustering Model Comparison
clustering_results = pd.DataFrame(results)

clustering_results = clustering_results.sort_values(
    by='Silhouette Score',
    ascending=False
)

clustering_results.round(4)